# Outliers Detection PATH method

--------------------

**Input**: 
* **routine DHIS2** data formatted.
    * from Dataset "**snt-dhis2-formatted**", `XXX_routine_data.parquet`

**Output**: 
All outputs saved to Dataset **snt-outliers-imputation**, with the following .parquet files:
* **outliers table** with flags for outlier data points:
    *  cols: YEAR, MONTH, ADM1_ID, ADM2_ID, OU_ID, INDICATOR, VALUE, **OUTLIER_TREND**
    *  Filename: `XXX_routine_outliers_detected.parquet`
* **Routine data imputed** Original routine data with imputed value:
    *   cols: PERIOD, YEAR, MONTH, ADM1_ID, ADM1_NAME, ADM2_ID, ADM2_NAME, OU_ID, OU_NAME, `DHIS2_INDICATORS`
    *   Filename: `XXX_routine_outliers_imputed.parquet`
* **Routine data removed** Original routine data with outliers values removed:
    *   cols: PERIOD, YEAR, MONTH, ADM1_ID, ADM1_NAME, ADM2_ID, ADM2_NAME, OU_ID, OU_NAME, `DHIS2_INDICATORS`
    *   Filename: `XXX_routine_outliers_removed.parquet`
* **DB Table** 🐘 in OpenHEXA WS **Database** with added cols needed for 📊 Shiny App: SNT Outliers Explorer
    *   cols: YEAR, MONTH, ADM1_ID, ADM2_ID, OU_ID, INDICATOR, VALUE, **OUTLIER_TREND**
    *   Table name: `outliers_detection_results`

---------------------

In [ ]:
# Parameters
ROOT_PATH <- "~/workspace"
DEVIATION_MEAN <- 10

## 1. Setup

In [ ]:
# Project folders (ROOT_PATH injected by pipeline if available)
if (!exists("ROOT_PATH")) ROOT_PATH <- "~/workspace"
PIPELINE_PATH <- file.path(ROOT_PATH, "pipelines", "snt_dhis2_outliers_imputation_path")

# Shared helpers for this pipeline (code)
source(file.path(PIPELINE_PATH, "utils", "snt_dhis2_outliers_imputation_path.r"))
setup_ctx <- bootstrap_path_context(
  root_path = ROOT_PATH,
  required_packages = c("arrow", "tidyverse", "jsonlite", "DBI", "RPostgres", "reticulate", "glue")
)

OUTPUT_DIR <- setup_ctx$OUTPUT_DIR

### 1.1. Validate parameters

In [ ]:
if (!exists("DEVIATION_MEAN")) DEVIATION_MEAN <- 10 

### 1.2. Load and check `SNT_config` file

In [ ]:
# Load SNT config from bootstrap context
config_json <- setup_ctx$config_json
log_msg(glue("SNT configuration loaded from  : {file.path(setup_ctx$CONFIG_PATH, 'SNT_config.json')}"))

In [ ]:
# Configuration validation is handled in pipeline.py
# Set config vars
COUNTRY_CODE <- config_json$SNT_CONFIG$COUNTRY_CODE
ADMIN_1 <- toupper(config_json$SNT_CONFIG$DHIS2_ADMINISTRATION_1)
ADMIN_2 <- toupper(config_json$SNT_CONFIG$DHIS2_ADMINISTRATION_2)

DHIS2_INDICATORS <- names(config_json$DHIS2_DATA_DEFINITIONS$DHIS2_INDICATOR_DEFINITIONS)  # PATH: c("TEST", "CONF", "PRES")

## 2. Load Data

### 2.1. **Routine** data (DHIS2) 

Formatted & aggregated data stored in OpenHEXA Dataset "**SNT_DHIS2_FORMATTED**"

In [ ]:
# Load file from dataset (formatting)
dataset_name <- config_json$SNT_DATASET_IDENTIFIERS$DHIS2_DATASET_FORMATTED
dhis2_routine <- load_routine_data(
  dataset_name = dataset_name,
  country_code = COUNTRY_CODE,
  required_indicators = DHIS2_INDICATORS
)

print(dim(dhis2_routine))
head(dhis2_routine, 4)

🔍 **Assert indicators are present**

In [ ]:
# Indicator validation is handled inside load_routine_data().

## 3. Outliers Detection

### 3.1. Transform routine data  

* **Pivot longer***: cols become rows

In [ ]:
# Helper loaded from utils/snt_dhis2_outliers_imputation_path.r
dhis2_routine_long <- build_path_routine_long(dhis2_routine, DHIS2_INDICATORS)

print(dim(dhis2_routine_long))
head(dhis2_routine_long, 2)

🔍 **Remove duplicated values**

In [ ]:
dedup_result <- remove_path_duplicates(dhis2_routine_long)
dhis2_routine_long <- dedup_result$data
duplicated <- dedup_result$duplicated
if (nrow(duplicated) > 0) {
    head(duplicated)
}

### 3.2. Calculate **summary stats**
At `OU_ID` (Health Facility) x `INDICATOR`, calculate:
* `MEAN_80` mean over center percentile 80%
* `SD_80` standard deviation over center percentile 80%

In [ ]:
# Compute and add mean_80 and sd_80 columns (PATH method)
log_msg(glue("Computing trend outliers (PATH Method) over: {paste(DHIS2_INDICATORS, collapse=', ')}."))

# Exclude zeros as these are more likely period of not reporting than true zeros.
routine_mean_80 <- dhis2_routine_long %>% 
    filter(VALUE > 0) %>%
    group_by(ADM1_ID, ADM2_ID, OU_ID, INDICATOR) %>%
    arrange(VALUE) %>%
    mutate(rank = row_number()) %>%
    # keep values with in the 10th and 90th percentiles (to reduce effects of outliers)
    filter(rank > n() * 0.1, rank <= n() * 0.9) %>% # Filter the middle 80%
    summarise(MEAN_80 = ceiling(mean(VALUE, na.rm = TRUE)), 
              SD_80 = ceiling(sd(VALUE, na.rm = TRUE)), .groups= "drop")

# join raw routine_data with corresponding mean value for each OU
dhis2_routine_stats <- dhis2_routine_long %>%
  left_join(routine_mean_80, by = c("ADM1_ID", "ADM2_ID", "OU_ID", "INDICATOR"))

### 3.3. Flag outliers

In [ ]:
# ⚠️ UPDATED: Added minimum value thresholds to exclude low values from outlier consideration
# unusually high values (outliers = True)
dhis2_routine_outliers <- dhis2_routine_stats %>%
  mutate(OUTLIER_TREND = case_when(VALUE > (MEAN_80 + DEVIATION_MEAN * SD_80) ~ TRUE, TRUE ~ FALSE)) %>%
  mutate(OUTLIER_TREND = case_when(is.na(VALUE) | is.na(SD_80) ~ FALSE, TRUE ~ OUTLIER_TREND)) %>%  # is.na(MEAN_80)?
  # ⚠️ NEW: Exclude too low values from outlier consideration (matching R script logic)
  mutate(OUTLIER_TREND = case_when(
    INDICATOR == "TEST" & VALUE < 50 ~ FALSE,
    INDICATOR == "PRES" & VALUE < 50 ~ FALSE,
    INDICATOR == "CONF" & VALUE < 10 ~ FALSE,
    TRUE ~ OUTLIER_TREND))

dim(dhis2_routine_outliers)
head(dhis2_routine_outliers, 2)

## 4. Exceptions

### 4.1. Detect possible stock-outs: 
1) If 'presumed cases' `(PRES)` suddenly jumps up at a time that testing (TEST), this indicates a RDT stockout - so add a condition where this isn't an outlier if it is within some range of the average number of confirmed cases.

In [ ]:
possible_stockout <- detect_possible_stockout(dhis2_routine_outliers, DEVIATION_MEAN)

In [ ]:
# Log possible stockouts
stockouts_n <- length(unique(possible_stockout$OU_ID))
if (stockouts_n > 0) {
    log_msg(glue("There are {length(unique(possible_stockout$OU_ID))} Health Facilities (OUs) with both high presumed cases (within a reasonable range) and low testing in the routine_data."))
}

head(possible_stockout, 2)

### 4.2. Detect possible epidemic: 

⚠️ **UPDATED LOGIC**

Sometimes tests and confirmed cases both jump up together - this can be due to a genuine epidemic event, or because there was a lag in reporting in previous months.

**New criteria:** Identify possible epidemics when:
- Confirmed cases (CONF) is an outlier, AND
- EITHER:
  - Tests (TEST) is also an outlier, OR
  - Total tests >= Total confirmed cases

In [ ]:
possible_epidemic <- detect_possible_epidemic(dhis2_routine_outliers, DEVIATION_MEAN)

epidemic_n <- length(unique(possible_epidemic$OU_ID))
if (epidemic_n > 0) {    
    print(glue("There are {epidemic_n} health facilities (OUs) where both the number of tests and confirmed cases increased sharply at the same time, sign of possible epidemic."))
}
head(possible_epidemic, 2)

### 4.3. Join exception tables and correct outlier flags

Join corrected outlier columns into the final table:

**OUTLIERS_TREND**: Indicates whether its outside a reasonable range.  
**OUTLIERS_TREND_01** (exception 1): Corrects the presumed cases when possible RDT or stockout.  
**OUTLIERS_TREND_02** (exception 2): Corrects potential epidemic.

In [ ]:
routine_data_outliers_clean <- build_path_clean_outliers(
    dhis2_routine_outliers = dhis2_routine_outliers,
    possible_stockout = possible_stockout,
    possible_epidemic = possible_epidemic
)

print(dim(routine_data_outliers_clean))
head(routine_data_outliers_clean, 2)

In [ ]:
# routine_data_outliers_clean %>% filter(OUTLIER_TREND==TRUE) %>% head(3)
# TEST VALUE : OU_ID=="a6ajNA9VZ7z" & PERIOD=="202201" (TEST) 52 -> 25.1

## 5. Routine data imputation

We consider the flagged outliers in the corrected column **OUTLIER_TREND_02** and we replace values by **MEAN_80**

### 5.1. Impute outliers with MEAN_80

⚠️ **UPDATED:** Added reversal check to prevent illogical corrections

After imputing outliers with MEAN_80, we check if the imputation created an impossible situation where:
- Imputed TEST < Imputed CONF (impossible - can't have more confirmed than tested)
- BUT Original TEST > Original CONF (the original data was logically consistent)

In such cases, we revert both TEST and CONF to their original values.

In [ ]:
routine_data_outliers_imputed <- impute_path_outliers(routine_data_outliers_clean)

print(dim(routine_data_outliers_imputed))
head(routine_data_outliers_imputed, 2)

In [ ]:
dim(routine_data_outliers_imputed[routine_data_outliers_imputed$OUTLIER_TREND==TRUE,])
dim(routine_data_outliers_imputed[routine_data_outliers_imputed$OUTLIER_TREND==FALSE,])

### 5.2. Format final `imputed` and `removed` routine data

**Imputed**: This table contains the routine data where outliers have been imputed.  
**Removed**: This table contains the routine data with outliers removed from the dataset.

In [ ]:
# get names from routine (This cleaning only applies to DRC names)
pyramid_names <- dhis2_routine %>% 
    distinct(ADM1_NAME, ADM1_ID, ADM2_NAME, ADM2_ID, OU_ID, OU_NAME) %>%
    # Simpify strings 
    mutate(
        ADM1_NAME = stringr::str_trim(str_remove_all(ADM1_NAME, "^[A-Z]{2}| PROVINCE")),
        ADM2_NAME = stringr::str_trim(str_remove_all(ADM2_NAME, "^[A-Z]{2}| ZONE DE SANTE"))
    )

In [ ]:
# Routine outliers imputed
dhis2_routine_outliers_imputed <- routine_data_outliers_imputed %>%
    select(-c("VALUE_OLD", "OUTLIER_TREND")) %>%
    pivot_wider(names_from = INDICATOR, values_from = VALUE_IMPUTED) %>%
    mutate(YEAR = as.integer(substr(PERIOD, 1, 4)), MONTH = as.integer(substr(PERIOD, 5, 6))) %>%
    left_join(pyramid_names, by = c("ADM1_ID", "ADM2_ID", "OU_ID")) %>% 
    select(all_of(c("PERIOD", 
                    "YEAR", 
                    "MONTH", 
                    "ADM1_NAME", 
                    "ADM1_ID", 
                    "ADM2_NAME", 
                    "ADM2_ID", 
                    "OU_ID", 
                    "OU_NAME", 
                    DHIS2_INDICATORS)))
    
print(dim(dhis2_routine_outliers_imputed))
head(dhis2_routine_outliers_imputed, 2)

In [ ]:
# Routine outliers removed
dhis2_routine_outliers_removed <- routine_data_outliers_imputed %>%
    filter(OUTLIER_TREND == FALSE) %>%
    select(-c("VALUE_OLD", "OUTLIER_TREND")) %>%
    pivot_wider(names_from = INDICATOR, values_from = VALUE_IMPUTED) %>%
    mutate(YEAR = as.integer(substr(PERIOD, 1, 4)), MONTH = as.integer(substr(PERIOD, 5, 6))) %>%
    filter(!if_all(all_of(DHIS2_INDICATORS), is.na)) %>%
    left_join(pyramid_names, by = c("ADM1_ID", "ADM2_ID", "OU_ID")) %>% 
    select(all_of(c("PERIOD", 
                    "YEAR", 
                    "MONTH", 
                    "ADM1_NAME", 
                    "ADM1_ID", 
                    "ADM2_NAME", 
                    "ADM2_ID", 
                    "OU_ID", 
                    "OU_NAME", 
                    DHIS2_INDICATORS))) 
    
print(dim(dhis2_routine_outliers_removed))
head(dhis2_routine_outliers_removed, 2)

In [ ]:
# log
nr_of_outliers <- nrow(routine_data_outliers_clean[routine_data_outliers_clean$OUTLIER_TREND == TRUE,])
perc_outliers <- nr_of_outliers/nrow(routine_data_outliers_clean) * 100
log_msg(glue("Using PATH outliers detection method {nr_of_outliers} outliers were identified ({sprintf('%.3f', perc_outliers)} % of values).")) 

## 6. Export Output tables
Export tables as .parquet files to `data/` folder

In [ ]:
output_path <- OUTPUT_DIR

# Save routine outliers table (parquet)
outliers_parquet <- file.path(output_path , paste0(COUNTRY_CODE, "_routine_outliers_detected.parquet")) 
routine_outliers_db_table <- routine_data_outliers_clean %>% 
    transmute(
      PERIOD, YEAR, MONTH, ADM1_ID, ADM2_ID, OU_ID, INDICATOR, VALUE,
      OUTLIER_DETECTED = OUTLIER_TREND,
      OUTLIER_METHOD = "PATH"
    ) %>%
    mutate(DATE = make_date(year = YEAR, month = MONTH, day = 1L)) %>%
    left_join(pyramid_names, by = c("ADM1_ID", "ADM2_ID", "OU_ID")) 

write_parquet(routine_outliers_db_table, outliers_parquet)
log_msg(glue("Outliers detection table saved under: {outliers_parquet}"))

In [ ]:
# Save routine data imputed (parquet)   
imputed_parquet <- file.path(output_path, paste0(COUNTRY_CODE, "_routine_outliers_imputed.parquet"))
write_parquet(dhis2_routine_outliers_imputed, imputed_parquet)
log_msg(glue("Routine data outliers imputed saved under: {imputed_parquet}"))

In [ ]:
# Save routine data removed (parquet)   
removed_parquet <- file.path(output_path , paste0(COUNTRY_CODE, "_routine_outliers_removed.parquet"))
write_parquet(dhis2_routine_outliers_removed, removed_parquet)
log_msg(glue("Routine data outliers removed saved under: {removed_parquet}"))